<p align="center"><img src="../docs/logo.jpg" alt="GIK-IceChain" width="480"/></p>

# End-to-End Pipeline Test — C1 → C2 → C3

**GIK-IceChain v2.0 — ECMWF Code for Earth 2026**

This notebook executes the full pipeline in a real environment and asserts correctness at each step.

| Part | Backend | When to use |
|------|---------|-------------|
| **Part 1** | Local MinIO | Local dev / CI — no AWS account needed |
| **Part 2** | Amazon S3 (production) | Final submission validation |

Execute cells **in order** within each part. A failing `assert` means the pipeline is broken.

---
## 0. Setup — install dependencies and import modules

In [ ]:
# Run once — installs all required packages
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-e", "../[cloud]",
     "boto3",
    ],
    check=True,
)
print("Dependencies installed.")

In [ ]:
import json
import os
import sys
import tempfile
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

# One-day window — enough to verify the full pipeline; change for longer runs
TEST_DATE_START = date(2024, 10, 15)
TEST_DATE_END   = date(2024, 10, 15)

print(f"Repo root : {REPO_ROOT}")
print(f"Test window: {TEST_DATE_START} → {TEST_DATE_END}")

In [ ]:
# ── Assertion helper ─────────────────────────────────────────────────────────
def check(condition: bool, msg: str) -> None:
    """Print a pass/fail banner and raise on failure."""
    if condition:
        print(f"  ✓  {msg}")
    else:
        print(f"  ✗  FAILED: {msg}")
        raise AssertionError(msg)


def section(title: str) -> None:
    print(f"\n{'─' * 60}")
    print(f"  {title}")
    print(f"{'─' * 60}")


print("Helpers loaded.")

---
## Part 1 — MinIO (local)

### Prerequisites

Start a MinIO container (one-time):

```bash
docker run -d --name minio-gik \
  -p 9000:9000 -p 9001:9001 \
  -e MINIO_ROOT_USER=minioadmin \
  -e MINIO_ROOT_PASSWORD=minioadmin123 \
  -v "$PWD/.minio-data:/data" \
  minio/minio server /data --console-address ":9001"
```

Console → http://localhost:9001  (minioadmin / minioadmin123)

### 1.1 Configure MinIO connection

In [ ]:
MINIO_ENDPOINT   = "http://localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
MINIO_BUCKET     = "gik-e2e-test"
MINIO_REGION     = "us-east-1"   # dummy — required by AWS SDK

# These are the URIs the pipeline will write to
MINIO_ICECHUNK_URI   = f"s3://{MINIO_BUCKET}/icechunk-store"
MINIO_EXCEEDANCE_URI = f"s3://{MINIO_BUCKET}/exceedance-zarr"
MINIO_RISK_URI       = f"s3://{MINIO_BUCKET}/admin1-risk"

# Point the AWS SDK and IceChunk/obstore at MinIO
os.environ["AWS_ENDPOINT_URL"]       = MINIO_ENDPOINT
os.environ["AWS_ACCESS_KEY_ID"]      = MINIO_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"]  = MINIO_SECRET_KEY
os.environ["AWS_DEFAULT_REGION"]     = MINIO_REGION
os.environ["GIK_ECMWF_ENDPOINT_URL"] = ""  # keep ECMWF reads on real AWS

print("MinIO environment set.")
print(f"  IceChunk store  : {MINIO_ICECHUNK_URI}")
print(f"  Exceedance store: {MINIO_EXCEEDANCE_URI}")

### 1.2 Create MinIO bucket

In [ ]:
import boto3
from botocore.config import Config

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name=MINIO_REGION,
    config=Config(signature_version="s3v4"),
)

try:
    s3.create_bucket(Bucket=MINIO_BUCKET)
    print(f"Bucket created: {MINIO_BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists: {MINIO_BUCKET}")

# Verify MinIO is reachable
buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
check(MINIO_BUCKET in buckets, f"Bucket '{MINIO_BUCKET}' visible in MinIO")

### 1.3 C1 — Ingest GIK Parquet → IceChunk virtual store

In [ ]:
section("C1: Ingest GIK → IceChunk")

from gik_icechain.conversion.gik_loader import GIKCatalog
from gik_icechain.conversion.icechunk_writer import IceChainStore
from gik_icechain.conversion.virtualizer import parquet_to_virtual_dataset

HF_DATASET = "E4DRR/gik-ecmwf-par"

catalog = GIKCatalog(HF_DATASET)
catalog.load_catalog()
print(f"Catalog loaded — HuggingFace dataset: {HF_DATASET}")

store = IceChainStore(MINIO_ICECHUNK_URI)
store.create_or_open()
print(f"IceChunk store ready: {MINIO_ICECHUNK_URI}")

last_commit = ""
current = TEST_DATE_START
while current <= TEST_DATE_END:
    paths = catalog.get_parquet_paths(
        start=current,
        end=current,
        run_hours=(0,),
        variables=["tp", "2t", "ro"],
    )
    if not paths:
        print(f"  No Parquet files for {current} — skipping")
        current += timedelta(days=1)
        continue

    print(f"  {current}: {len(paths)} Parquet files")
    vds = parquet_to_virtual_dataset(paths, variables=["tp", "2t", "ro"])
    print(f"    Virtual dataset: {dict(vds.dims)}  vars={list(vds.data_vars)}")

    last_commit = store.commit_day(current, vds, run_hour=0)
    print(f"    IceChunk commit: {last_commit[:12]}")
    current += timedelta(days=1)

print(f"\nLast commit: {last_commit[:12] if last_commit else 'none'}")

In [ ]:
section("C1 assertions")

snapshots = store.list_snapshots()
committed_dates = [s["forecast_date"] for s in snapshots if s["forecast_date"]]

check(len(committed_dates) >= 1, "At least one day committed to IceChunk")
check(TEST_DATE_START.isoformat() in committed_dates,
      f"Test date {TEST_DATE_START} present in IceChunk store")
check(last_commit != "", "commit hash is non-empty")
check(len(last_commit) > 8, f"commit hash looks valid (len={len(last_commit)})")

# Verify the virtual dataset dimensions are correct
check("step" in vds.dims or "latitude" in vds.dims,
      "Virtual dataset has expected spatial/temporal dims")
check("tp" in vds.data_vars, "Variable 'tp' present in virtual dataset")

print(f"\nSnapshots in store: {len(snapshots)}")
print(f"Committed dates:    {committed_dates}")

In [ ]:
section("C1 — Time-travel checkout (IceChunk Innovation 1)")

ds_historical = store.checkout_as_of(TEST_DATE_START)
print(f"Checked out as-of {TEST_DATE_START}:")
print(f"  dims : {dict(ds_historical.dims)}")
print(f"  vars : {list(ds_historical.data_vars)}")

check(isinstance(ds_historical, xr.Dataset), "Time-travel returns xr.Dataset")
check(len(ds_historical.data_vars) > 0, "Historical dataset is non-empty")

### 1.4 C2 — Adaptive GEV exceedance probabilities

In [ ]:
section("C2: Exceedance probabilities")

import pandas as pd
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations
from gik_icechain.exceedance.exceedance import (
    compute_ensemble_confidence,
    compute_exceedance_probabilities,
)
from gik_icechain.exceedance.thresholds import (
    AdaptiveGEVThresholds,
    ClimateMode,
    ENSOPhase,
    IODPhase,
    classify_enso,
    classify_iod,
    get_season,
)
from gik_icechain.exceedance.writer import build_exceedance_dataset, write_exceedance_store

THRESHOLDS_PATH = REPO_ROOT / "data" / "cmorph_thresholds"
ENSO_IOD_PATH   = REPO_ROOT / "data" / "enso_iod_index.csv"
WINDOWS_H       = [24, 72, 168]
RETURN_PERIODS  = [5, 20]

thresholds = AdaptiveGEVThresholds.load(THRESHOLDS_PATH)
print(f"Thresholds loaded: {len(thresholds._thresholds)} climate modes")

enso_iod = pd.read_csv(ENSO_IOD_PATH, parse_dates=["date"]).set_index("date")
print(f"ENSO/IOD index: {len(enso_iod)} rows")

In [ ]:
results      = {}
conf_results = {}

store2 = IceChainStore(MINIO_ICECHUNK_URI)
store2.create_or_open()
session = store2._repo.readonly_session(branch=store2.branch)

for date_str in committed_dates:
    day = date.fromisoformat(date_str)
    month = day.month
    season = get_season(month)

    try:
        row = enso_iod.loc[pd.Timestamp(day)]
        enso = classify_enso(float(row["nino34"]))
        iod  = classify_iod(float(row["dmi"]))
    except KeyError:
        from gik_icechain.exceedance.thresholds import ENSOPhase, IODPhase
        enso, iod = ENSOPhase.NEUTRAL, IODPhase.NEUTRAL

    mode = ClimateMode(season, enso, iod)
    print(f"  {date_str}  mode={mode.key}")

    day_ds = xr.open_zarr(session.store, group=date_str, consolidated=False)
    day_ds = day_ds.chunk({"member": -1, "step": -1, "latitude": 50, "longitude": 50})

    acc_ds = compute_rolling_accumulations(day_ds, windows_h=WINDOWS_H)
    print(f"    Accumulations: {list(acc_ds.data_vars)}")

    day_results = {}
    for w in WINDOWS_H:
        for rp in RETURN_PERIODS:
            try:
                thr = thresholds.get(w, rp, mode)
                p = compute_exceedance_probabilities(
                    acc_ds,
                    xr.Dataset({f"rp_{rp}y": thr}),
                    window_h=w,
                    return_period=rp,
                    member_dim="member",
                )
                day_results[(w, rp)] = p
            except Exception as exc:
                print(f"    skip w={w} rp={rp}: {exc}")

    if day_results:
        results[day] = build_exceedance_dataset(day_results, day)
        print(f"    Exceedance: {len(day_results)} (window, rp) combinations")

    try:
        conf = compute_ensemble_confidence(acc_ds, window_h=24, member_dim="member")
        conf_results[day] = conf.assign_coords(date=pd.Timestamp(day)).expand_dims("date")
        print(f"    Confidence states: {np.unique(conf.values)}")
    except Exception as exc:
        print(f"    Confidence skipped: {exc}")

print(f"\nDays with exceedance results: {len(results)}")

In [ ]:
section("C2 — Write exceedance store to MinIO")

write_exceedance_store(
    results,
    MINIO_EXCEEDANCE_URI,
    append=True,
    confidence_dict=conf_results or None,
)
print(f"Written to: {MINIO_EXCEEDANCE_URI}")

In [ ]:
section("C2 assertions")

exc_ds = xr.open_zarr(MINIO_EXCEEDANCE_URI, consolidated=False)
print("Exceedance store schema:")
print(exc_ds)

check("exceedance_prob" in exc_ds, "Variable 'exceedance_prob' present")
check("ensemble_confidence" in exc_ds,
      "Variable 'ensemble_confidence' present (Data_Confidence BN node)")
check("date" in exc_ds.dims, "Dimension 'date' present")
check("window" in exc_ds.dims, "Dimension 'window' present")
check("return_period" in exc_ds.dims, "Dimension 'return_period' present")

p_vals = exc_ds["exceedance_prob"].values
p_finite = p_vals[np.isfinite(p_vals)]
check(len(p_finite) > 0, "At least some finite exceedance probability values")
check(float(p_finite.min()) >= 0.0, "All exceedance probabilities >= 0")
check(float(p_finite.max()) <= 1.0, "All exceedance probabilities <= 1")

conf_vals = exc_ds["ensemble_confidence"].values
unique_conf = set(int(v) for v in conf_vals.flatten() if np.isfinite(v))
check(unique_conf.issubset({0, 1, 2}), f"Confidence states ⊆ {{0,1,2}}: got {unique_conf}")

print(f"\nExceedance store dims: {dict(exc_ds.dims)}")
print(f"Mean P(exceed 24h/5y): {float(exc_ds['exceedance_prob'].sel(window=24, return_period=5).mean()):.3f}")

### 1.5 C3 — CRMA Bayesian Network risk inference

In [ ]:
section("C3: CRMA risk inference")

import geopandas as gpd
from gik_icechain.risk.crma_model import CRMAModel
from gik_icechain.risk.risk_engine import run_risk_batch

ADMIN_PATH  = REPO_ROOT / "data" / "admin_boundaries" / "east_africa_admin1.gpkg"
GPM_DIR     = REPO_ROOT / "data" / "gpm_imerg"
RISK_OUT    = Path(tempfile.mkdtemp()) / "minio_risk"

print(f"Admin boundaries: {ADMIN_PATH}")
print(f"GPM IMERG dir:    {GPM_DIR}")
print(f"Risk output:      {RISK_OUT}")

crma = CRMAModel()
crma.build()
print("CRMA model built.")

written = run_risk_batch(
    exceedance_store_uri=MINIO_EXCEEDANCE_URI,
    gpm_dir=GPM_DIR,
    admin_boundaries_path=ADMIN_PATH,
    crma_model=crma,
    output_dir=RISK_OUT,
    start=TEST_DATE_START,
    end=TEST_DATE_END,
)

print(f"\nGeoJSON files written: {len(written)}")
for p in written:
    print(f"  {p.name}  ({p.stat().st_size // 1024} KB)")

In [ ]:
section("C3 assertions")

check(len(written) >= 1, f"At least one GeoJSON file written")

geojson_path = written[0]
fc = json.loads(geojson_path.read_text())
features = fc["features"]

check(fc["type"] == "FeatureCollection", "Output is a valid GeoJSON FeatureCollection")
check(len(features) > 0, f"FeatureCollection has {len(features)} features")

sample = features[0]["properties"]
required_props = {"admin1_pcode", "risk_label", "risk_state", "p_green", "p_yellow", "p_orange", "p_red"}
check(required_props.issubset(sample.keys()),
      f"All required properties present: {required_props}")

risk_states  = [f["properties"]["risk_state"] for f in features]
risk_labels  = [f["properties"]["risk_label"]  for f in features]
valid_states = {0, 1, 2, 3}
valid_labels = {"Green", "Yellow", "Orange", "Red"}

check(set(risk_states).issubset(valid_states),
      f"risk_state values ⊆ {{0,1,2,3}}: got {set(risk_states)}")
check(set(risk_labels).issubset(valid_labels),
      f"risk_label values ⊆ {{Green/Yellow/Orange/Red}}: got {set(risk_labels)}")

# Probability sums must be 1
for f in features[:5]:
    p = f["properties"]
    total = p["p_green"] + p["p_yellow"] + p["p_orange"] + p["p_red"]
    check(abs(total - 1.0) < 1e-4,
          f"{p['admin1_pcode']}: P(Green+Yellow+Orange+Red) = {total:.4f} ≈ 1")

print(f"\nSample admin-1 results ({written[0].name}):")
print(f"{'pcode':<12} {'risk':<8} {'p_red':<8} {'p_orange':<8}")
for f in features[:8]:
    p = f["properties"]
    print(f"{p['admin1_pcode']:<12} {p['risk_label']:<8} {p['p_red']:.3f}    {p['p_orange']:.3f}")

### 1.6 Extra verifications — IceChunk store health

In [ ]:
section("IceChunk store validation")

report = store.validate()
print(json.dumps(report, indent=2))

check(report["committed_days"] >= 1, "Store has at least one committed day")
check(report["gaps_detected"] == 0, f"No date gaps detected")
check("tp" in report["variables_present"], "Variable 'tp' present in latest snapshot")

In [ ]:
section("IceChunk monthly compaction")

compact_result = store.compact(keep_days=30)
print(f"Compaction result: {compact_result}")

check(isinstance(compact_result["expired"], int),
      "compact() returns dict with 'expired' integer count")

In [ ]:
section("Consolidated Zarr thresholds — save_zarr / load_zarr round-trip")

ZARR_THRESHOLD_URI = f"s3://{MINIO_BUCKET}/thresholds-consolidated.zarr"

thresholds.save_zarr(ZARR_THRESHOLD_URI)
print(f"Saved consolidated thresholds to: {ZARR_THRESHOLD_URI}")

thr2 = AdaptiveGEVThresholds.load_zarr(ZARR_THRESHOLD_URI)
print(f"Loaded back: {len(thr2._thresholds)} climate modes")

check(len(thr2._thresholds) == len(thresholds._thresholds),
      "Round-trip preserves the number of climate modes")

# Spot-check one value
from gik_icechain.exceedance.thresholds import Season
mode_check = ClimateMode(Season.OND, ENSOPhase.NEUTRAL, IODPhase.NEUTRAL)
orig = thresholds.get(24, 5, mode_check)
rt   = thr2.get(24, 5, mode_check)
max_diff = float(np.abs(orig.values - rt.values).max())
check(max_diff < 1e-3, f"24h/5y threshold round-trip max diff = {max_diff:.2e} mm")

### 1.7 MinIO summary

In [ ]:
section("Part 1 summary — MinIO")

print("All assertions passed.")
print()
print(f"  IceChunk commits    : {report['committed_days']}")
print(f"  Exceedance days     : {len(results)}")
print(f"  Risk GeoJSON files  : {len(written)}")
print(f"  Admin-1 units       : {len(features)}")
print(f"  Risk state distribution:")
from collections import Counter
dist = Counter(f["properties"]["risk_label"] for f in features)
for label, count in sorted(dist.items()):
    bar = "█" * (count * 30 // len(features))
    print(f"    {label:<8} {count:>4}  {bar}")

---
## Part 2 — Amazon S3 (production)

### Prerequisites

Set these environment variables **before running** this section:

```bash
export AWS_ACCESS_KEY_ID=AKIA...
export AWS_SECRET_ACCESS_KEY=...
export AWS_DEFAULT_REGION=eu-west-1
export GIK_ICECHUNK_STORE_URI=s3://your-bucket/gik-icechain-store
export GIK_EXCEEDANCE_STORE_URI=s3://your-bucket/exceedance-zarr
export GIK_BUCKET=your-bucket
```

### 2.1 Configure production credentials

In [ ]:
# Remove MinIO endpoint so calls go to real AWS
for key in ["AWS_ENDPOINT_URL", "GIK_ECMWF_ENDPOINT_URL"]:
    os.environ.pop(key, None)

missing = [v for v in [
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "GIK_ICECHUNK_STORE_URI",
    "GIK_EXCEEDANCE_STORE_URI",
    "GIK_BUCKET",
] if not os.environ.get(v)]

if missing:
    print("SKIP — the following env vars are not set:")
    for v in missing:
        print(f"  {v}")
    print("\nSet them and re-run Part 2.")
    PROD_SKIP = True
else:
    PROD_SKIP = False
    S3_ICECHUNK_URI   = os.environ["GIK_ICECHUNK_STORE_URI"]
    S3_EXCEEDANCE_URI = os.environ["GIK_EXCEEDANCE_STORE_URI"]
    S3_BUCKET         = os.environ["GIK_BUCKET"]
    S3_RISK_URI       = f"s3://{S3_BUCKET}/admin1-risk"

    print("Production credentials set:")
    print(f"  IceChunk store  : {S3_ICECHUNK_URI}")
    print(f"  Exceedance store: {S3_EXCEEDANCE_URI}")

### 2.2 C1 — Ingest to production IceChunk store

In [ ]:
if PROD_SKIP:
    print("Skipped (credentials not set).")
else:
    section("C1 — Production S3 ingest")

    prod_store = IceChainStore(S3_ICECHUNK_URI)
    prod_store.create_or_open()
    print(f"Store ready: {S3_ICECHUNK_URI}")

    prod_catalog = GIKCatalog(HF_DATASET)
    prod_catalog.load_catalog()

    prod_commit = ""
    current = TEST_DATE_START
    while current <= TEST_DATE_END:
        paths = prod_catalog.get_parquet_paths(
            start=current, end=current,
            run_hours=(0,), variables=["tp", "2t", "ro"],
        )
        if not paths:
            print(f"  No files for {current}")
            current += timedelta(days=1)
            continue

        vds_prod = parquet_to_virtual_dataset(paths, variables=["tp", "2t", "ro"])
        prod_commit = prod_store.commit_day(current, vds_prod, run_hour=0)
        print(f"  {current}: commit {prod_commit[:12]}")
        current += timedelta(days=1)

    prod_snapshots = prod_store.list_snapshots()
    prod_dates = [s["forecast_date"] for s in prod_snapshots if s["forecast_date"]]

    check(TEST_DATE_START.isoformat() in prod_dates,
          f"Test date committed to production store")
    print(f"  Committed dates in production store: {prod_dates}")

### 2.3 C2 — Exceedance computation on production store

In [ ]:
if PROD_SKIP:
    print("Skipped.")
else:
    section("C2 — Production S3 exceedance")

    prod_results = {}
    prod_conf    = {}

    prod_sess = prod_store._repo.readonly_session(branch=prod_store.branch)

    for date_str in prod_dates:
        day = date.fromisoformat(date_str)
        season = get_season(day.month)
        try:
            row  = enso_iod.loc[pd.Timestamp(day)]
            enso = classify_enso(float(row["nino34"]))
            iod  = classify_iod(float(row["dmi"]))
        except KeyError:
            enso, iod = ENSOPhase.NEUTRAL, IODPhase.NEUTRAL

        mode   = ClimateMode(season, enso, iod)
        day_ds = xr.open_zarr(prod_sess.store, group=date_str, consolidated=False)
        day_ds = day_ds.chunk({"member": -1, "step": -1, "latitude": 50, "longitude": 50})
        acc_ds = compute_rolling_accumulations(day_ds, windows_h=WINDOWS_H)

        day_results = {}
        for w in WINDOWS_H:
            for rp in RETURN_PERIODS:
                try:
                    thr = thresholds.get(w, rp, mode)
                    p   = compute_exceedance_probabilities(
                        acc_ds, xr.Dataset({f"rp_{rp}y": thr}),
                        window_h=w, return_period=rp, member_dim="member",
                    )
                    day_results[(w, rp)] = p
                except Exception as exc:
                    print(f"    skip w={w} rp={rp}: {exc}")

        if day_results:
            prod_results[day] = build_exceedance_dataset(day_results, day)

        try:
            conf = compute_ensemble_confidence(acc_ds, window_h=24, member_dim="member")
            prod_conf[day] = conf.assign_coords(date=pd.Timestamp(day)).expand_dims("date")
        except Exception:
            pass

    write_exceedance_store(
        prod_results, S3_EXCEEDANCE_URI,
        append=True,
        confidence_dict=prod_conf or None,
    )

    prod_exc = xr.open_zarr(S3_EXCEEDANCE_URI, consolidated=False)
    check("exceedance_prob"     in prod_exc, "exceedance_prob written to S3")
    check("ensemble_confidence" in prod_exc, "ensemble_confidence written to S3")
    print(f"  Production exceedance store dims: {dict(prod_exc.dims)}")

### 2.4 C3 — CRMA risk inference on production data

In [ ]:
if PROD_SKIP:
    print("Skipped.")
else:
    section("C3 — Production S3 risk inference")

    PROD_RISK_OUT = Path(tempfile.mkdtemp()) / "prod_risk"

    prod_written = run_risk_batch(
        exceedance_store_uri=S3_EXCEEDANCE_URI,
        gpm_dir=GPM_DIR,
        admin_boundaries_path=ADMIN_PATH,
        crma_model=crma,
        output_dir=PROD_RISK_OUT,
        start=TEST_DATE_START,
        end=TEST_DATE_END,
    )

    check(len(prod_written) >= 1, "At least one GeoJSON file written from production data")

    prod_fc  = json.loads(prod_written[0].read_text())
    prod_fts = prod_fc["features"]

    check(len(prod_fts) > 0, f"Production GeoJSON has {len(prod_fts)} features")

    for f in prod_fts[:5]:
        p = f["properties"]
        total = p["p_green"] + p["p_yellow"] + p["p_orange"] + p["p_red"]
        check(abs(total - 1.0) < 1e-4, f"{p['admin1_pcode']}: probabilities sum to 1")

    print(f"\nProduction GeoJSON features: {len(prod_fts)}")
    dist = Counter(f["properties"]["risk_label"] for f in prod_fts)
    for label, count in sorted(dist.items()):
        bar = "█" * (count * 30 // len(prod_fts))
        print(f"  {label:<8} {count:>4}  {bar}")

### 2.5 Production store validation

In [ ]:
if PROD_SKIP:
    print("Skipped.")
else:
    section("Production IceChunk store validation")

    prod_report = prod_store.validate()
    print(json.dumps(prod_report, indent=2))

    check(prod_report["committed_days"] >= 1, "Production store has committed days")
    check(prod_report["gaps_detected"] == 0,  "No date gaps in production store")
    check("tp" in prod_report["variables_present"],
          "Variable 'tp' present in production store")

---
## Final summary

In [ ]:
print("=" * 60)
print("  GIK-IceChain E2E Pipeline Test — Final Summary")
print("=" * 60)
print()
print(f"  Test window    : {TEST_DATE_START} → {TEST_DATE_END}")
print()
print("  Part 1 — MinIO")
print(f"    C1 IceChunk commits : {report['committed_days']}")
print(f"    C2 exceedance days  : {len(results)}")
print(f"    C3 GeoJSON files    : {len(written)}")
print(f"    Admin-1 units       : {len(features)}")
print()
if not PROD_SKIP:
    print("  Part 2 — Amazon S3 (production)")
    print(f"    C1 commits          : {prod_report['committed_days']}")
    print(f"    C2 exceedance days  : {len(prod_results)}")
    print(f"    C3 GeoJSON files    : {len(prod_written)}")
else:
    print("  Part 2 — Amazon S3   : SKIPPED (credentials not set)")
print()
print("  All assertions passed.")
print("=" * 60)